# 07 · Export for the app

Writes what the app needs into `export/` (committed, so the app can take it without running the notebooks):

| File | Contents |
| --- | --- |
| `export/presets.json` | The presets chosen in notebook 05, with their rules in the app's rule format and plain-language titles. **Internal**: households never see or choose a preset (week 9); the questionnaire assigns it |
| `export/onboarding.json` | The whole questionnaire: the splash text, the onboarding tree (stage 1), each leaf's preset and its follow-up questions (stage 2, the "help me refine my settings" wizard), every question with its full wording and examples per device class, and how answers become rules |
| `export/onboarding-tree.svg` | A picture of the onboarding tree, for the README |

**How a household's rules are made** (and remade): walk the tree with the household's answers to a preset, copy the
preset's rules, then write every *yes* / *no* the household gave to a situation (on the path and in the follow-ups)
into those rules (below). Rerunning the questionnaire **starts over**: the rules are rebuilt from the new answers and
manual edits are replaced (the app warns first).

The notebook then **checks the exported files themselves**, the way the app would read them: the rules reproduce each
preset exactly, writing each household's answers into the rules gives exactly notebook 05's result, exactly one rule
decides every flow, and every term and condition is valid PPD.

In [1]:
import json
from datetime import date
from pathlib import Path
from textwrap import wrap
from xml.sax.saxutils import escape
import numpy as np, pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA, EXPORT = ROOT / "data", ROOT / "export"
items = pd.read_csv(DATA / "items.csv")
cells = pd.read_csv(DATA / "cells.csv")
D = np.load(DATA / "decisions.npy")
answers = pd.read_csv(DATA / "answers.csv", dtype=str)
QUESTIONS = {q["id"]: q for q in json.loads((DATA / "questions.json").read_text())}
tree = json.loads((DATA / "onboarding_tree.json").read_text())
K = tree["k"]
presets = json.loads((DATA / tree["presets_file"]).read_text())
preset_items = np.load(DATA / "presets" / f"preset_items_k{K}.npy")
FIELDS = ["data_type", "purpose", "action", "source", "handling_context"]
TERMS = {f: sorted(items[f].unique()) for f in FIELDS}
item_cell = items.merge(cells[["cell_id", "data_type", "purpose", "handling_context"]],
                        on=["data_type", "purpose", "handling_context"], how="left").cell_id.to_numpy()
real = items.in_corpus.to_numpy()
ORDINAL = np.array([0.0, 1.0, 0.5, 0.5, 0.5])
n = len(D)
print(f"exporting k = {K} presets, the depth-{tree['depth']} tree and its follow-ups (at most {tree['total_questions_cap']} questions)")

exporting k = 4 presets, the depth-6 tree and its follow-ups (at most 15 questions)


## Plain language

Every core term gets a short label households can read (rule titles), and every answer option a label.

In [2]:
PLAIN = {
    "ppd:identifierData": "account and identity details", "ppd:profileData": "profile and preferences",
    "ppd:sensorData": "sensor readings", "ppd:contentData": "video, photos and sound", "ppd:locationData": "location",
    "ppd:deviceAndNetworkData": "device and network details", "ppd:usageData": "how you use your devices",

    "ppd:coreFunctionality": "making features work", "ppd:personalization": "personalizing what you see",
    "ppd:communicationsAndNotifications": "alerts and messages", "ppd:security": "security",
    "ppd:diagnosticsAndMaintenance": "fixing problems", "ppd:analyticsAndImprovement": "improving products",
    "ppd:legalCompliance": "legal obligations", "ppd:advertisingAndMarketing": "advertising",

    "ppd:collection": "collected", "ppd:use": "used", "ppd:transfer": "sent on", "ppd:inference": "used to work out new information",

    "ppd:userProvided": "you entered", "ppd:participantObserved": "the device observes", "ppd:participantGenerated": "the device generates",
    "ppd:householdProvided": "other devices in your home provide", "ppd:vendorProvided": "the maker provides",
    "ppd:thirdPartyProvided": "other companies provide", "ppd:derivedFromPriorData": "worked out from earlier data",

    "ppd:householdContext": "staying in your home", "ppd:vendorContext": "going to the device maker",
    "ppd:thirdPartyContext": "going to other companies", "ppd:publicAudience": "made public",
}
SHORT = {   # for the presets' internal names
    "ppd:advertisingAndMarketing": "ads", "ppd:analyticsAndImprovement": "product improvement", "ppd:personalization": "personalization",
    "ppd:communicationsAndNotifications": "alerts", "ppd:diagnosticsAndMaintenance": "fixing problems", "ppd:coreFunctionality": "features",
    "ppd:security": "security", "ppd:legalCompliance": "legal duties",
    "ppd:thirdPartyContext": "other companies", "ppd:publicAudience": "anything public", "ppd:vendorContext": "the maker", "ppd:householdContext": "keeping it home",
    "ppd:contentData": "video", "ppd:locationData": "location", "ppd:identifierData": "identity", "ppd:profileData": "profiles",
    "ppd:sensorData": "sensor data", "ppd:usageData": "usage data", "ppd:deviceAndNetworkData": "device details",
}
EFFECT_TEXT = {0: "Never", 1: "Allow", 2: "Allow only if not kept", 3: "Allow only if processed in your home", 4: "Allow only if processed on the device"}
OPTION_LABEL = {"no": "No", "depends": "It depends", "yes": "Yes", "with_limits": "Only with limits"}
assert set(PLAIN) == {t for f in FIELDS for t in TERMS[f]}, "every core term needs a plain label"

## Rule titles

A rule's scope lists the terms it covers per field; a field covering most of its terms reads better as "any … except".
Example: *Never: video, photos and sound · going to other companies · for anything except making features work*.

In [3]:
def terms_text(field, terms):
    everything = TERMS[field]
    if len(terms) == len(everything):
        return None
    missing = [t for t in everything if t not in terms]
    if field == "handling_context":
        if set(terms) == set(everything) - {"ppd:householdContext"}:
            return "leaving your home"
        return " or ".join(PLAIN[t] for t in terms)
    words = {"data_type": ("", "any data except "), "purpose": ("for ", "for anything except "),
             "action": ("when ", "except when "), "source": ("that ", "unless ")}[field]
    if len(missing) < len(terms):
        return words[1] + ", ".join(PLAIN[t] for t in missing) if field != "action" else "except when " + " or ".join(PLAIN[t] for t in missing)
    return words[0] + (" or ".join(PLAIN[t] for t in terms) if field in ("action", "source") else ", ".join(PLAIN[t] for t in terms))

def decision_of(rule):
    if rule["effect"] == "dont_allow":
        return 0
    return {None: 1, "retention=ppd:ephemeral": 2, "processing_boundary=ppd:inHomeOnly": 3, "processing_boundary=ppd:onDeviceOnly": 4}[rule["condition"]]

def rule_title(rule):
    scope = {f: rule["scope"].get(f, TERMS[f]) for f in FIELDS}
    order = ["data_type", "handling_context", "purpose", "action", "source"]
    parts = [p for f in order if (p := terms_text(f, scope[f]))]
    return f"{EFFECT_TEXT[decision_of(rule)]}: " + (" · ".join(parts) if parts else "everything else")

## Presets (internal)

Households never see a preset's name or choose a preset (week 9: a list invites self-perception bias, e.g. picking the
strictest and finding everything blocked). Each preset keeps an internal working name and a description for the team
and for generated explanations: how strict it is, and where it is stricter or more open than most households.

In [4]:
overall = ORDINAL[D].mean(axis=0)
def describe(i):
    acc = ORDINAL[preset_items[i]]
    deltas = {}
    for f in ["purpose", "handling_context", "data_type"]:
        deltas.update(pd.Series(acc - overall).groupby(items[f]).mean().to_dict())
    s = pd.Series(deltas).sort_values()
    strict = [SHORT[t] for t in s.index[:2] if s[t] < -0.05]
    relaxed = [SHORT[t] for t in s.index[::-1][:2] if s[t] > 0.05]
    level = acc.mean()
    tone = "Strict" if level < 0.45 else "Careful" if level < 0.6 else "Relaxed" if level > 0.75 else "Balanced"
    name = tone + (f" · no {' or '.join(strict)}" if strict else "") + (f" · OK with {' and '.join(relaxed)}" if relaxed else "")
    allow, cond, deny = (preset_items[i] == 1).mean(), np.isin(preset_items[i], [2, 3, 4]).mean(), (preset_items[i] == 0).mean()
    text = (f"Allows {allow:.0%} of possible data flows, {cond:.0%} only with conditions, and refuses {deny:.0%}. "
            + (f"Stricter than most households about {', '.join(strict)}" if strict else "Close to most households")
            + (f"; more open about {', '.join(relaxed)}." if relaxed else "."))
    return name, text

def leaves(node):
    return [node] if "preset" in node else [l for b in node["branches"] for l in leaves(b["next"])]
reached = {l["preset"] for l in leaves(tree["root"])}

out_presets = []
for i, p in enumerate(presets["presets"]):
    name, text = describe(i)
    rules = []
    for j, r in enumerate(p["rules"]):
        rule = {"id": f"{p['id']}-r{j}", "effect": r["effect"], "condition": r["condition"],
                "scope": r["scope"], "enabled": True, "origin": {"kind": "preset", "preset_id": p["id"]}}
        rule["title"] = rule_title(rule)
        rules.append(rule)
    out_presets.append({"id": p["id"], "internal_name": name, "description": text, "shown_to_households": False,
                        "accepted_share": round(float((preset_items[i] != 0).mean()), 3),
                        "households_share": round(p["households"] / n, 3), "reached_by_onboarding": p["id"] in reached,
                        "rules": rules})
pd.DataFrame([{k: p[k] for k in ["id", "internal_name", "accepted_share", "households_share", "reached_by_onboarding"]}
              | {"rules": len(p["rules"])} for p in out_presets]).set_index("id")

,internal_name,accepted_share,households_share,reached_by_onboarding,rules
id,,,,,
k4-p0,Strict · no fixing problems or legal duties,0.224,0.192,True,30
k4-p1,Strict · no product improvement or personaliza...,0.406,0.347,True,30
k4-p2,Relaxed · OK with legal duties and fixing prob...,0.816,0.151,True,30
k4-p3,Careful · no ads · OK with security and device...,0.600,0.310,True,30


## Writing answers into the rules

A preset's rules do not overlap: exactly one decides each flow. A situation (data type × purpose × where the data
goes) is a block of 28 flows. When the household answers **yes** or **no** to it and the rule covering it says
otherwise, that rule is **split** around the situation, keeping everything else it decided, and a rule with the
household's answer is added for the situation (*It depends* keeps the preset). Splitting a rule's block around one
situation gives at most three pieces: the other data types; this data type with the other purposes; this data type
and purpose with the other places the data goes. Rules stay non-overlapping, so no rule has to "win" over another,
and the rules page shows one consistent list, each rule marked as a starting rule or as coming from an answer.

Every situation the household answered counts, on the onboarding path and in the follow-ups: it said what it wants.

In [5]:
ANSWER_DECISION = {"yes": 1, "no": 0}
full = lambda scope: {f: list(scope.get(f, TERMS[f])) for f in FIELDS}
compact = lambda scope: {f: v for f, v in scope.items() if len(v) < len(TERMS[f])}

def write_answers(rules, answered):
    """rules: a preset's rules; answered: [(question id, answer)]. Returns the household's rules."""
    rules = [dict(r, scope=full(r["scope"])) for r in rules]
    for qid, ans in answered:
        if ans not in ANSWER_DECISION or "cell" not in QUESTIONS[qid]:
            continue
        d, cell = ANSWER_DECISION[ans], QUESTIONS[qid]["cell"]
        t, p, c = cell["data_type"], cell["purpose"], cell["handling_context"]
        out = []
        for r in rules:
            s = r["scope"]
            if not (t in s["data_type"] and p in s["purpose"] and c in s["handling_context"]) or decision_of(r) == d:
                out.append(r)
                continue
            pieces = [{**s, "data_type": [x for x in s["data_type"] if x != t]},
                      {**s, "data_type": [t], "purpose": [x for x in s["purpose"] if x != p]},
                      {**s, "data_type": [t], "purpose": [p], "handling_context": [x for x in s["handling_context"] if x != c]}]
            out += [dict(r, scope=piece) for piece in pieces if all(piece[f] for f in FIELDS)]
            out.append({"effect": "allow" if d == 1 else "dont_allow", "condition": None, "enabled": True,
                        "scope": {**s, "data_type": [t], "purpose": [p], "handling_context": [c]},
                        "origin": {"kind": "answer", "question": qid}})
        rules = out
    for j, r in enumerate(rules):
        r["scope"] = compact(r["scope"]); r["title"] = rule_title(r); r["id"] = f"r{j}"
    return rules

## The questionnaire document

The tree is pruned of questions whose answers all lead to the same place (same preset *and* same follow-ups). Every
question the tree or a follow-up uses is listed once with its full wording, options and, for situations, an example
per device class (week 9: show an example from a device the household owns).

In [6]:
def prune(node):
    if "preset" in node:
        return {k: node[k] for k in ["preset", "follow_up_budget", "follow_ups"]}
    branches = [{"answers": b["answers"], "next": prune(b["next"])} for b in node["branches"]]
    ends = {json.dumps(b["next"], sort_keys=True) for b in branches}
    return branches[0]["next"] if len(ends) == 1 else {"question": node["question"], "branches": branches}

root = prune(tree["root"])
used = []
def collect(node):
    if "preset" in node:
        used.extend(node["follow_ups"])
    else:
        used.append(node["question"])
        for b in node["branches"]:
            collect(b["next"])
collect(root)

def question_doc(qid):
    q = QUESTIONS[qid]
    doc = {"prompt": q["text"], "options": [{"id": o, "label": OPTION_LABEL.get(o, o)} for o in q["options"]]}
    if "cell" in q:
        doc["situation"] = q["cell"]
        doc["data_examples"] = q["data_examples"]
    return doc

source = {"generated": date.today().isoformat(), "by": "ppd-household-presets notebooks 01-07",
          "data": "synthetic households (not survey data); see data/DATACARD.md",
          "k": K, "depth": tree["depth"], "total_questions_cap": tree["total_questions_cap"],
          "match_all_items_cv": {"one_preset": tree["one_preset_all_cv"], "after_profiling": tree["match_all_cv"],
                                 "after_follow_ups": tree["match_all_after_follow_ups_cv"]},
          "match_real_device_flows_cv": {"one_preset": tree["one_preset_real_cv"], "after_profiling": tree["match_real_cv"],
                                         "after_follow_ups": tree["match_real_after_follow_ups_cv"]}}
onboarding_doc = {
    "version": "3.0.0", "kind": "tree+follow_ups", "source": source,
    "splash": {"title": "Set up your home's privacy rules",
               "text": ("We'll show you up to 15 short situations, like a camera sending video to its maker. Tell us whether you'd "
                        "allow each one. From your answers we set up rules for every smart device in your home, before any of them "
                        "joins. It takes about three minutes, you can change any rule later, and you can redo this at any time.")},
    "follow_up_intro": {"title": "Help us get the details right",
                        "text": "Households like yours are split on these situations, so we'd rather ask than guess."},
    "questions": {q: question_doc(q) for q in dict.fromkeys(used)},
    "walk": ("Stage 1: show the root question; follow the branch whose 'answers' contain the chosen option; repeat until a node "
             "has 'preset'. A question can come back further down a path: reuse the earlier answer, never ask twice. "
             "Stage 2: ask the leaf's 'follow_ups' in order, at most 'follow_up_budget' of them (the cap minus the questions "
             "asked in stage 1). Skip any follow-up already answered in stage 1."),
    "rules": ("Copy the preset's rules from presets.json. For every situation question answered 'yes' or 'no' (stage 1 or 2), "
              "where the rule covering that situation says otherwise, split that rule around the situation (at most three "
              "pieces: other data types; this data type with other purposes; this data type and purpose with other handling "
              "contexts) and add a rule with the answer for the situation itself ('yes' = allow, 'no' = never; origin "
              "kind 'answer'). 'It depends' keeps the preset. Rules never overlap, so exactly one decides each flow."),
    "rerun": "Rerunning the questionnaire starts over: the household's rules are rebuilt from the new answers; manual edits are replaced after a warning.",
    "root": root,
}
presets_doc = {
    "version": "2.0.0", "source": source, "max_rules": presets["max_rules"],
    "rule_format": ("Like the app's Rule (shared/types.ts): scope lists core ppd: terms per field (a missing field means any), "
                    "effect allow or dont_allow; 'condition' on an allow rule is a PPD qualifier the compiler sends as constraints "
                    "(protocol-04 7.11-7.12); origin kind 'preset' or 'answer'. A preset's rules never overlap."),
    "presets": out_presets,
}
print(f"{len(onboarding_doc['questions'])} questions in the document "
      f"({sum('situation' in q for q in onboarding_doc['questions'].values())} situations), {len(leaves(root))} leaves after pruning")

68 questions in the document (68 situations), 57 leaves after pruning


## Check the exported files

These checks read only the documents just built, the way the app would. For every household: walk the exported tree,
take the exported preset, write its answers into the rules, and compare with notebook 05's result (the preset's
decisions with every answered situation set to the answer).

In [7]:
TERM_MASK = {(f, t): items[f].eq(t).to_numpy() for f in FIELDS for t in TERMS[f]}
def covers(rule):
    m = np.ones(len(items), dtype=bool)
    for f, terms in rule["scope"].items():
        m &= np.logical_or.reduce([TERM_MASK[(f, t)] for t in terms])
    return m

def apply_rules(rules):
    out, count = np.full(len(items), -1), np.zeros(len(items), dtype=int)
    for r in rules:
        m = covers(r)
        out[m] = decision_of(r); count += m
    return out, count

ids = [p["id"] for p in out_presets]
cell_items = {int(c): np.where(item_cell == c)[0] for c in np.unique(item_cell)}
cell_of_q = {qid: int(cells[(cells.data_type == q["cell"]["data_type"]) & (cells.purpose == q["cell"]["purpose"])
                            & (cells.handling_context == q["cell"]["handling_context"])].cell_id.iloc[0])
             for qid, q in QUESTIONS.items() if "cell" in q}

def walk(row):
    node, path = onboarding_doc["root"], []
    while "preset" not in node:
        if node["question"] not in path:
            path.append(node["question"])
        node = next(b["next"] for b in node["branches"] if row[node["question"]] in b["answers"])
    follow = [q for q in node["follow_ups"] if q not in path][:node["follow_up_budget"]]
    return node["preset"], path, follow

applied = {p["id"]: apply_rules(p["rules"]) for p in out_presets}
mismatch, overlaps, n_rules, n_answer_rules, asked = 0, 0, [], [], []
for h, row in enumerate(answers.to_dict("records")):
    pid, path, follow = walk(row)
    answered = [(q, row[q]) for q in path + follow]
    rules = write_answers(next(p for p in out_presets if p["id"] == pid)["rules"], answered)
    got, count = apply_rules(rules)
    expected = preset_items[ids.index(pid)].copy()
    for q, a in answered:
        if q in cell_of_q and a in ANSWER_DECISION:
            expected[cell_items[cell_of_q[q]]] = ANSWER_DECISION[a]
    mismatch += int((got != expected).any()); overlaps += int((count != 1).any())
    n_rules.append(len(rules)); n_answer_rules.append(sum(r["origin"]["kind"] == "answer" for r in rules)); asked.append(len(path) + len(follow))
    if h == 1:
        example = (pid, answered, rules)
n_rules, asked = np.array(n_rules), np.array(asked)

def branches_ok(node):
    if "preset" in node:
        return node["preset"] in ids and all(q in onboarding_doc["questions"] for q in node["follow_ups"])
    opts = [o["id"] for o in onboarding_doc["questions"][node["question"]]["options"]]
    covered = [a for b in node["branches"] for a in b["answers"]]
    return sorted(covered) == sorted(opts) and all(branches_ok(b["next"]) for b in node["branches"])

checks = {
    "exported rules give exactly each preset's decisions on all items": all((applied[p["id"]][0] == preset_items[i]).all() for i, p in enumerate(out_presets)),
    "exactly one rule decides every flow, in every preset": all((applied[p["id"]][1] == 1).all() for p in out_presets),
    "every term is a core ppd: term of its field": all(set(t) <= set(TERMS[f]) for p in out_presets for r in p["rules"] for f, t in r["scope"].items()),
    "in-home / on-device conditions only on use and inference": all(
        set(r["scope"].get("action", TERMS["action"])) <= {"ppd:use", "ppd:inference"}
        for p in out_presets for r in p["rules"] if (r["condition"] or "").startswith("processing_boundary")),
    "every preset keeps to its rule count": all(len(p["rules"]) <= presets["max_rules"] for p in out_presets),
    "every tree branch covers each answer exactly once and ends in a known preset": branches_ok(onboarding_doc["root"]),
    "every preset is reached by the onboarding": reached == set(ids),
    "no household is asked more than the cap": asked.max() <= tree["total_questions_cap"],
    "writing each household's answers into the rules gives exactly notebook 05's result": mismatch == 0,
    "after the answers, exactly one rule still decides every flow": overlaps == 0,
    "children are never asked about (week 9)": "children" not in onboarding_doc["questions"],
}
print(f"households' rules after their answers: {n_rules.mean():.0f} on average (preset alone: {np.mean([len(p['rules']) for p in out_presets]):.0f}), "
      f"at most {n_rules.max()}; {np.mean(n_answer_rules):.1f} rules from answers on average; questions asked: {asked.mean():.1f} on average, at most {asked.max()}")
assert all(checks.values()), checks
pd.Series(checks, name="holds")

households' rules after their answers: 43 on average (preset alone: 30), at most 66; 9.7 rules from answers on average; questions asked: 15.0 on average, at most 15


exported rules give exactly each preset's decisions on all items                      True
exactly one rule decides every flow, in every preset                                  True
every term is a core ppd: term of its field                                           True
in-home / on-device conditions only on use and inference                              True
every preset keeps to its rule count                                                  True
every tree branch covers each answer exactly once and ends in a known preset          True
every preset is reached by the onboarding                                             True
no household is asked more than the cap                                               True
writing each household's answers into the rules gives exactly notebook 05's result    True
after the answers, exactly one rule still decides every flow                          True
children are never asked about (week 9)                                               True

## Preview: one household from start to end

In [8]:
pid, answered, rules = example
p = next(x for x in out_presets if x["id"] == pid)
print(f"preset {pid} ({p['internal_name']}, internal): {p['description']}\n")
print("answers to situations:")
for q, a in answered:
    if "cell" in QUESTIONS[q]:
        print(f"  {OPTION_LABEL.get(a, a):10} {QUESTIONS[q]['text'][:120]}…")
print(f"\nits rules ({len(rules)}; {sum(r['origin']['kind'] == 'answer' for r in rules)} from its answers):")
for r in rules:
    print(f"  {'★' if r['origin']['kind'] == 'answer' else ' '} {r['title']}")

preset k4-p3 (Careful · no ads · OK with security and device details, internal): Allows 60% of possible data flows, 0% only with conditions, and refuses 40%. Stricter than most households about ads; more open about security, device details.

answers to situations:
  Yes        A device sends how you use your devices (for example, when you switch the lights on, or what you watch on TV) to the com…
  No         A device's maker passes your account details (for example, your name, email address and account ID) on to other companie…
  No         A device sends video or audio (for example, a camera's video of your living room, or what you say to a voice assistant) …
  Yes        A device's maker passes your location (for example, your phone's location, to know when you are home or away) on to othe…
  No         A device's maker passes your location (for example, your phone's location, to know when you are home or away) on to othe…
  No         A device's maker passes video or audio (for exa

## Save

In [9]:
EXPORT.mkdir(exist_ok=True)
(EXPORT / "presets.json").write_text(json.dumps(presets_doc, indent=2, ensure_ascii=False) + "\n")
(EXPORT / "onboarding.json").write_text(json.dumps(onboarding_doc, indent=2, ensure_ascii=False) + "\n")
print("wrote", [f.name for f in sorted(EXPORT.iterdir())])

wrote ['onboarding-tree.svg', 'onboarding.json', 'presets.json']


## Picture of the onboarding tree

`export/onboarding-tree.svg`, for the README: stage 1 questions left to right (shortened to *data → where it goes,
why*), the answers on the arrows, and the preset each path ends in, numbered from the strictest (1) to the most open;
each path then continues with its follow-ups (stage 2), listed in `onboarding.json`.

In [10]:
SD = {"identifierData": "Account details", "profileData": "Profile of you", "sensorData": "Sensor readings", "contentData": "Video/audio",
      "locationData": "Location", "deviceAndNetworkData": "Device & network", "usageData": "How you use devices"}
SW = {"householdContext": "stays home", "vendorContext": "→ maker", "thirdPartyContext": "→ other companies", "publicAudience": "→ public"}
SY = {"coreFunctionality": "so it works", "personalization": "to personalize", "communicationsAndNotifications": "to send alerts",
      "security": "for security", "diagnosticsAndMaintenance": "to fix problems", "analyticsAndImprovement": "to improve products",
      "legalCompliance": "for legal duties", "advertisingAndMarketing": "for ads"}
def short_q(qid):
    q = onboarding_doc["questions"][qid]
    if "situation" not in q:
        return q["prompt"]
    c = {f: v.split(":")[1] for f, v in q["situation"].items()}
    return f"{SD[c['data_type']]} {SW[c['handling_context']]}, {SY[c['purpose']]}?"

order = sorted(range(len(out_presets)), key=lambda i: -(preset_items[i] == 0).mean())
RANK = {out_presets[i]["id"]: r + 1 for r, i in enumerate(order)}
BY_ID = {p["id"]: p for p in out_presets}
COLOUR = lambda a: ("#FCE8E6", "#B3261E") if a < 0.45 else ("#FEF7E0", "#8E6A00") if a < 0.6 else ("#E8F0FE", "#0B57D0") if a < 0.75 else ("#E6F4EA", "#1E7D34")

COL, ROW, QW, LW, TOP, LEFT = 330, 44, 200, 230, 95, 20
def depth(nd):
    return 0 if "preset" in nd else 1 + max(depth(b["next"]) for b in nd["branches"])
DEPTH = depth(root)
nodes, edges, row = [], [], [0]
def place(nd, d):
    if "preset" in nd:
        y = TOP + row[0] * ROW; row[0] += 1
        nodes.append(("leaf", nd["preset"], LEFT + DEPTH * COL, y))
        return y
    labels = {o["id"]: o["label"] for o in onboarding_doc["questions"][nd["question"]]["options"]}
    kids = [(place(b["next"], d + 1), " / ".join(labels[a] for a in b["answers"]), "preset" in b["next"]) for b in nd["branches"]]
    y = sum(cy for cy, *_ in kids) / len(kids)
    nodes.append(("q", nd["question"], LEFT + d * COL, y))
    edges.extend((d, cy, label, to_leaf, y) for cy, label, to_leaf in kids)
    return y
place(root, 0)

def svg_text(x, y, lines, size=12, weight=400, colour="#1F1F1F"):
    y0 = y - (len(lines) - 1) * (size + 2) / 2
    return "".join(f'<text x="{x}" y="{y0 + i * (size + 2) + size / 3:.1f}" font-size="{size}" font-weight="{weight}" fill="{colour}">{escape(line)}</text>'
                   for i, line in enumerate(lines))

svg = []
for d, cy, label, to_leaf, py in edges:
    x1, x2 = LEFT + d * COL + QW, (LEFT + DEPTH * COL) if to_leaf else LEFT + (d + 1) * COL
    mid = x1 + 24
    svg.append(f'<path d="M{x1},{py:.1f} C{mid},{py:.1f} {mid},{cy:.1f} {mid + 16},{cy:.1f} L{x2},{cy:.1f}" fill="none" stroke="#9AA0A6" stroke-width="1.2"/>')
    svg.append(f'<text x="{mid + 20}" y="{cy - 5:.1f}" font-size="11" fill="#3C4043" stroke="#FFFFFF" stroke-width="3" paint-order="stroke">{escape(label)}</text>')
for kind, key, x, y in nodes:
    if kind == "q":
        lines = wrap(short_q(key), 26); h = len(lines) * 14 + 12
        svg.append(f'<rect x="{x}" y="{y - h / 2:.1f}" width="{QW}" height="{h}" rx="8" fill="#FFFFFF" stroke="#5F6368" stroke-width="1.1"/>')
        svg.append(svg_text(x + 9, y, lines, 12, 500))
    else:
        p = BY_ID[key]; fill, ink = COLOUR(p["accepted_share"])
        svg.append(f'<rect x="{x}" y="{y - 15}" width="{LW}" height="30" rx="15" fill="{fill}" stroke="{ink}" stroke-width="1.1"/>')
        svg.append(svg_text(x + 12, y, [f"{RANK[key]} · {p['internal_name'].split(' · ')[0]} ({p['accepted_share']:.0%} accepted)"], 12, 600, ink))
W, H = LEFT * 2 + DEPTH * COL + LW, TOP + row[0] * ROW + 10
header = (svg_text(LEFT, 28, [f"Onboarding stage 1: at most {tree['depth']} situations place the household in one of {K} presets "
                               f"(1 = strictest); then up to {tree['total_questions_cap']} questions in total with the follow-ups (stage 2)"], 15, 600)
          + svg_text(LEFT, 52, ["Situations are shortened to 'data → where it goes, why'. The full wording with examples is in export/onboarding.json."], 12, 400, "#5F6368"))
doc = (f'<svg xmlns="http://www.w3.org/2000/svg" width="{W}" height="{H}" viewBox="0 0 {W} {H}" font-family="Helvetica, Arial, sans-serif">'
       f'<rect width="{W}" height="{H}" fill="#FFFFFF"/>{header}{"".join(svg)}</svg>')
(EXPORT / "onboarding-tree.svg").write_text(doc + "\n")
print(f"wrote onboarding-tree.svg: {sum(1 for k_, *_ in nodes if k_ == 'q')} questions, {row[0]} end points, {W}x{H}")

wrote onboarding-tree.svg: 56 questions, 57 end points, 2250x2613


## For the app

What `ppd-preference-app` needs to use these files:

1. **Splash, then stage 1.** Show `splash`, then walk `root` (questions in `questions`, options with labels), showing
   an example from a device class the household owns (`data_examples`; `default` otherwise).
2. **Stage 2, the "help me refine my settings" wizard.** Ask the leaf's `follow_ups` (at most `follow_up_budget`) with
   `follow_up_intro`; it can also be rerun later as a privacy checkup. Never ask at the moment a device joins.
3. **No preset list.** Presets are internal (`shown_to_households: false`); the household never sees or picks one.
4. **Rules.** Build the household's rules as described in `rules` (split, never override): no rule precedence change is
   needed, the engine's "strictest matching rule wins" still holds because rules never overlap. Rules gain
   `condition?: string` on allow rules (sent as PPD `constraints`) and origin kinds `preset` and `answer`.
5. **Rerun = start over** (`rerun`): rebuild the rules from the new answers, after warning that manual edits are replaced.
6. **Children** are not asked about (week 9: future work).